<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>Simulation-Based Inference - Pre-Tutorial Reading</h1>

<h2>Structure</h2>

| Part | Topic | Type |
|:----:|-------|------|
| 0 | Why Simulation-Based Inference? | Motivation |
| 1 | Approximate Bayesian Computation | Conceptual + Code |
| 2 | Neural SBI (NPE, NLE, NRE) | Conceptual |
| 3 | Neural Density Estimators | Conceptual + Code |
| 3.3 | Mixture Density Networks | Math + Code |
| 4 | Normalising Flows | Math + Code |
| 5 | SBI in Practice: the <code>sbi</code> package | Hands-on |

<h2>What this notebook covers</h2>

We build the intuition for simulation-based inference from the ground up, starting from ABC and its limitations, through neural density estimators (MDNs and normalising flows), to a practical worked example using the <code>sbi</code> Python package. By the end, you will (hopefully) understand what SBI does, why it works, and when to reach for it.

<b>Estimated reading time:</b> 45–60 minutes (including running code cells).

<h2>Preamble</h2>

<ul>
<li>This notebook builds the conceptual and practical foundation for simulation-based inference. You will read and run this before the in-person tutorial.</li>
<li>You will not need to write code from scratch, only run it.</li>
<li>The ABC section (§1) runs with numpy only. The MDN and flow sections (§3–§4) require PyTorch. The practical section (§5) requires the <code>sbi</code> package.</li>
<li>Key references: Cranmer, Brehmer &amp; Louppe (2020) PNAS; Papamakarios et al. (2021) JMLR (normalising flows review); Boelts et al. (2025) <code>sbi</code> documentation.</li>
</ul>

<h2>Library Imports</h2>
Please make sure that you have the following packages imported before the tutorial. You can install packages within jupyter cells by running <code>!pip install sbi</code> for example.
</div>

In [ ]:
# Standard imports
import numpy as np
import matplotlib.pyplot as plt

from tqdm.notebook import tqdm

print('Standard imports done')

# ML stuff (PyTorch based)
import torch
import torch.nn as nn
import zuko

print('ML imports done')

# SBI package
from sbi.inference import NPE
from sbi.utils import BoxUniform
from sbi.analysis import pairplot

print('SBI imports done')

# Set seeds
np.random.seed(42)
torch.manual_seed(42)


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
<h1>§0. Why Simulation-Based Inference?</h1>
    
As we have discussed previously, Bayesian statistics is the <i>only</i> correct way to do statistics. Despite Bayesianism existing, in some sense, for over three hundred years, only somewhat recently has it been widely adopted. For several centuries, it was the correct framework, yet it was not used because it was completely impractical: simply too computationally demanding. 
    
With increasing computational power, Bayesian methods have become more popular. Techniques like MCMC (to approximate the posterior distribution) and nested sampling (to approximate the evidence) became available.

Despite the meteoric rise of computational power, there are problems that are still too computationally expensive for repeated likelihood evaluations. Consider galaxy formation simulations, such as IllustrisTNG or EAGLE. A single forward evaluation takes a supercomputer months, while MCMC would need millions of evaluations to converge on a posterior. Another case would be fitting a radiative transfer model to an exoplanet transmission spectrum: you <i>can</i> write down a likelihood, but each evaluation is so expensive that MCMC is impractical.

A separate failure mode can be found by looking at Bayes' formula:
$$P(\theta | D) = \frac{\mathcal{L}(\theta)\pi(\theta)}{\mathcal{Z}} $$
In order to get a posterior, we must be able to calculate the likelihood, $\mathcal{L}(\theta)$, and the prior, $\pi(\theta)$. To calculate these, we must first be able to write them down. Unfortunately, that is not always easy or possible. 

It can be impossible to write down a prior. In my current research (mapping stellar surfaces), I am trying to enforce priors over a spherical harmonic representation of the surface. However, there is no known way of even enforcing the surface to be positive (flux must be positive), let alone contain compact spots. 

It can also be impossible to write down a likelihood. Consider the problem of inferring the parameters of a stellar population synthesis code. The code can produce a synthetic spectrum from a stellar mass, age, metallicity, dust attenuation curve, star-formation history, etc. How does one write down a closed-form likelihood for that? A stellar population synthesis code works as such:
$$\mathbf{\theta} \rightarrow \text{SPS code} \rightarrow\mathbf{x} $$
The code will give you a spectrum for any $\theta$ you hand it, but it does not give you the likelihood $p(\mathbf{x} | \mathbf{\theta})$, because writing that density down requires marginalising over every stochastic choice the simulator made.

Classical Bayesian inference, in all of the above cases, is either impossible or so computationally intractable that you end up approximating the problem so aggressively that the "likelihood" you wrote down barely resembles the physics. In cases like these, a potential solution is simulation-based inference (also known as likelihood-free inference). The idea is that if you can <i>simulate</i> data from your model, that is enough.

<details class="alert alert-info" style="margin: 10px 0;">
<summary style="cursor: pointer; font-weight: bold;">Aside: We usually lie about our likelihood</summary>
<div style="margin-top: 10px;">

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Most Bayesian analyses in astrophysics use a likelihood of the form

$$\ln \mathcal{L}(\theta) = -\tfrac{1}{2} \sum_i \frac{(x_i - \mu_i(\theta))^2}{\sigma_i^2} + \text{const.}$$

This is a Gaussian likelihood with independent noise. Using it implicitly assumes:

<ol>
<li>The noise is Gaussian: It often isn't. Photon counts are Poisson, detector reads have non-Gaussian tails, and cosmic rays exist.</li>
<li>The noise is independent across data points: It often isn't. Pixels on the same CCD share readout noise, spectral pixels share continuum residuals, and time-series photometry has correlated stellar variability (See the Gaussian Processes to see why this is a problem).</li>
<li>The reported $\sigma_i$ is correct: It needs to include systematics, calibration uncertainty, or model misspecification.</li>
<li>The model $\mu(\theta)$ is correct. It's usually a tractable approximation, chosen because it's tractable.</li>
</ol>

Most of the time this is good enough. Posterior shrinkage is reasonably robust to modest misspecification, and you can remove the worst offenders by cutting data. But when you say "likelihood", be aware that you usually mean "the wrong likelihood that was convenient to write down".

SBI gets around this because you never write the likelihood down. You run the simulator. Whatever the simulator does defines $p(\mathbf{x} \mid \theta)$ implicitly. If the simulator injects real, non-Gaussian, non-stationary detector noise into synthetic signals, the neural network learns whatever that distribution is.

BUT: If the simulator is wrong, SBI gives you a wrong posterior. The old problem (writing down a bad likelihood) is replaced by a new one (writing down a bad simulator). The useful thing about this trade is that the simulator is usually a forward model of the physics and the instrument, which tends to be something you can think about physically. A bad Gaussian likelihood with made-up error bars is harder to interrogate.

The following phrase comes up a lot: <b>the simulator is the model</b>. Any approximation in your simulator is an approximation in your inference. We'll return to this in the diagnostics.

</div>
</details>


</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§0.1 Briefly: How SBI Works</h2>

You have a simulator that takes parameters $\theta$ and produces synthetic data $\mathbf{x}$. You run this simulator many times with parameters drawn from the prior, building a training set of $(\theta, \mathbf{x})$ pairs. You then train a neural network to learn the mapping from data to posterior. Specifically, in <b>Neural Posterior Estimation</b> (NPE), the network learns to output a conditional density estimate $q_\phi(\theta | \mathbf{x}) \approx p(\theta | \mathbf{x})$ directly.

The benefit is that you never write down a likelihood or prior. The prior is defined implicitly by whatever distribution you drew the training $\theta$ from. The likelihood is defined implicitly by whatever the simulator does.

<h2>§0.2 Amortisation</h2>

There is a property of NPE that has no analogue in classical Bayesian inference: amortisation.

When you run MCMC or nested sampling, all of the computational work is specific to one dataset. If you observe a new target, you start from scratch. For problems where the likelihood is cheap, this is fine. But consider large scale surveys with millions of observations. Running MCMC individually on each target is, at best, painful.

An amortised estimator pays the cost once, up front. The neural network is trained on simulated $(\theta, \mathbf{x})$ pairs drawn from across the full prior. Once trained, obtaining a full posterior for a new observation is a single forward pass through the network. This takes milliseconds. The training can take hours or days, but it only happens once.

The trade-off is that amortised inference can be less accurate than a bespoke MCMC run for any single observation, because the network must generalise across the full prior volume rather than concentrate on one point. In practice, the accuracy is usually sufficient, and if it isn't for a particular high-value target, you can always fall back to running MCMC on that one object (assuming the likelihood is tractable).

<h2>§0.3 What about the intractable prior / likelihood?</h2>

Return to the two problems from §0 that had nothing to do with computational cost:

<ol>
<li><b>Intractable prior</b> (e.g. enforcing positivity and compact spots on a spherical harmonic stellar surface). In NPE, the prior is defined by whatever distribution you sample $\theta$ from during training. If you can write a function that draws valid stellar surfaces (rejection sampling from the harmonic coefficients, a generative model, a physically motivated procedure), those draws <i>are</i> the prior.</li>
<li><b>Intractable likelihood</b> (e.g. stellar population synthesis). The simulator <i>is</i> the likelihood. Each $(\theta, \mathbf{x})$ pair is a sample from the joint $p(\theta, \mathbf{x}) = p(\mathbf{x}|\theta)\pi(\theta)$. The neural network learns the conditional $p(\theta|\mathbf{x})$ from these samples without ever requiring $p(\mathbf{x}|\theta)$ in closed form.</li>
</ol>

This is why SBI is sometimes called likelihood-free inference, as "it doesn't have a likelihood". The name is slightly misleading as there <i>is</i> a likelihood, it is just defined implicitly by the simulator rather than written as an equation. 

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
<h2>§1. ABC: Approximate Bayesian Computation</h2>

The first attempt at likelihood-free inference was conceptually simple. Suppose you want samples from the posterior $p(\theta \mid \mathbf{x}_\text{obs})$. Do the following:

<ol>
<li>Draw $\theta \sim \pi(\theta)$ from the prior.</li>
<li>Simulate $\mathbf{x} \sim p(\mathbf{x} \mid \theta)$.</li>
<li>If $\mathbf{x} = \mathbf{x}_\text{obs}$, keep $\theta$. Otherwise, discard it.</li>
<li>Repeat.</li>
</ol>

The surviving $\theta$ values are exact draws from the posterior.

The problem is obvious: if $\mathbf{x}$ is continuous or high-dimensional, the probability of exact equality is zero, so this never accepts anything.

<h3>§1.1 Making it work</h3>
Rather than waiting for exact equality, you instead accept draws within a small region of $\mathbf{x}$
<ol>
<li>Draw $\theta \sim \pi(\theta)$.</li>
<li>Simulate $\mathbf{x} \sim p(\mathbf{x} \mid \theta)$.</li>
<li>Compute a summary statistic $s(\mathbf{x})$.</li>
<li>Accept $\theta$ if $\|s(\mathbf{x}) - s(\mathbf{x}_\text{obs})\| < \varepsilon$.</li>
</ol>

This is Approximate Bayesian Computation (ABC). Two things to notice:

<ol>
<li>Instead of requiring exact match, we require closeness via a summary statistic. This introduces two approximations: (a) the summary statistic throws away information (unless it is sufficient, which it usually isn't), and (b) the tolerance $\varepsilon$ smooths the posterior. Setting $\varepsilon \to 0$ recovers the exact algorithm from §1.1, which never accepts.</li>
<li>It scales badly with dimensionality. If $\mathbf{x}$ is $d$-dimensional, the acceptance rate goes like $\varepsilon^d$. You need exponentially many simulations to maintain a given tolerance as $d$ grows.</li>
</ol>

ABC is still used where the data has already been aggressively compressed (e.g. cosmology with $n$-point statistics). But for high-dimensional raw data (a spectrum, a light curve, a density field) it is largely superseded by the neural methods we will get to.

<h3>§1.2 A problem with the summary statistic</h3>

The obvious move when ABC scales badly in $d$ is to compress the data. Pick a low-dimensional summary statistic $s(\mathbf{x})$ and run ABC on that. Two problems:

<ol>
<li>Picking $s$ well is hard. You want $s$ to be <i>sufficient</i> (meaning it retains all information about $\theta$). Outside of a few special cases (e.g. exponential families), sufficient statistics do not exist. So any $s$ you pick throws away some information. You are doing inference on a lossy compression of your data.</li>
<li>Handcrafting $s$ does not scale. For a transit light curve, you might pick depth, duration, ingress time. For a spectrum, you might pick equivalent widths, line ratios. Every new problem needs a new set of hand-designed summaries.</li>
</ol>

This is where ABC sat for a while. Any progress required either a way to do likelihood-free inference directly on high-dimensional data, or a way to learn summary statistics from simulations automatically.

------------


Let's have a play with ABC on a toy model. We will infer the mean $\mu$ of a Gaussian with a known variance $\sigma$. Here, we can compute the true posterior, but this is just for comparison. We will compare ABC when we choose a sufficient summary statistic (in this case the mean is all we need, as a the variance is fixed and a Gaussian is fully defined by its mean and variance), and show what happens when we choose a lossy summary statistic. We will also look at changing the value of $\varepsilon$.

For the record, a Gaussian likelihood with known $\sigma$ and a flat prior, the posterior on $\mu$ is:
$$\mu \,|\, \mathbf{X} \sim \mathcal{N}\left(\bar{x}, \frac{\sigma}{\sqrt{n}} \right) $$
</div>

In [ ]:
################################
# Ground Truth & Observed Data #
################################
np.random.seed(117)

mu_true = 2.5
sigma   = 1
n_obs   = 20
x_obs   = np.random.normal(mu_true, sigma, size = n_obs)


#####################
# Summary statistic #
#####################

s_obs = np.mean(x_obs)

######################
# Analytic posterior #
######################

posterior_mean = s_obs
posterior_std  = sigma / np.sqrt(n_obs)

#############################
# Do ABC rejection sampling #
#############################

n_sims  = int(5e5)
epsilon = 0.1 # tolerance

# Step 1:
mu_proposals = np.random.uniform(-10, 10, size = n_sims) # draw from prior
accepted = []

for i in tqdm(range(n_sims)):
    # Step 2:
    x_sim = np.random.normal(mu_proposals[i], sigma, size = n_obs) # simulate data
    # Step 3:
    s_sim = np.mean(x_sim) # calculate summary statistic

    # Step 4: Accept / Reject
    if np.abs(s_sim - s_obs) < epsilon:
        accepted.append(mu_proposals[i])


accepted = np.array(accepted)
print(f"Accepted {len(accepted)} / {n_sims} ({100*len(accepted)/n_sims:.2f}%)")

In [ ]:
########################################
# Plot ABC posterior vs True posterior #
########################################
mu_grid = np.linspace(0, 5, 500)
analytic_posterior = (1 / (posterior_std * np.sqrt(2 * np.pi))) * \
                     np.exp(-0.5 * ((mu_grid - posterior_mean) / posterior_std)**2)

fig, ax = plt.subplots(figsize=(8, 4))

# True posterior
ax.plot(mu_grid, analytic_posterior,
        'k-', lw=2, label='Analytic posterior')

# ABC histogram
ax.hist(accepted, bins=50, density=True, alpha=0.5, color='tab:blue',
        label=rf'ABC ($\varepsilon = {epsilon}$, $n = {len(accepted)}$)')

ax.axvline(mu_true, color='red', ls='--', alpha=0.5, label=r'True $\mu$')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel('Density')
ax.legend()
ax.set_title('ABC with a sufficient summary statistic (sample mean)')
plt.tight_layout()
plt.show()


In [ ]:
###############################################
# Now break it: use a BAD summary statistic   #
###############################################

# Instead of the sample mean (sufficient), use the median (not sufficient)
# and the max (also not sufficient, and very noisy)

summary_stats = {
    'Mean (sufficient)': lambda x: np.mean(x),
    'Median':            lambda x: np.median(x),
    'Max':               lambda x: np.max(x),
}

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, (name, stat_func) in zip(axes, summary_stats.items()):
    s_obs_this = stat_func(x_obs)
    
    accepted_this = []
    for i in tqdm(range(n_sims)):
        x_sim = np.random.normal(mu_proposals[i], sigma, size=n_obs)
        s_sim = stat_func(x_sim)
        if np.abs(s_sim - s_obs_this) < epsilon:
            accepted_this.append(mu_proposals[i])
    
    accepted_this = np.array(accepted_this)
    
    ax.plot(mu_grid, analytic_posterior,
            'k-', lw=2, label='True posterior')
    ax.hist(accepted_this, bins=50, density=True, alpha=0.5, color='tab:blue',
            label=f'ABC ({len(accepted_this)} accepted)')
    ax.axvline(mu_true, color='red', ls='--', alpha=0.5)
    ax.set_xlabel(r'$\mu$')
    ax.set_title(f'Summary: {name}')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Density')
fig.suptitle(r'ABC with different summary statistics — same $\varepsilon$, same data', fontsize=13)
plt.tight_layout()
plt.show()


In [ ]:
########################
# Messing with Epsilon #                  
########################

epsilons = [2.0, 0.5, 0.1, 0.02]

fig, axes = plt.subplots(1, len(epsilons), figsize=(16, 4), sharey=True)

for ax, eps in zip(axes, epsilons):
    accepted_eps = []
    for i in tqdm(range(n_sims)):
        x_sim = np.random.normal(mu_proposals[i], sigma, size=n_obs)
        s_sim = np.mean(x_sim)
        if np.abs(s_sim - s_obs) < eps:
            accepted_eps.append(mu_proposals[i])
    
    accepted_eps = np.array(accepted_eps)
    
    ax.plot(mu_grid, analytic_posterior,
            'k-', lw=2, label='True posterior')
    ax.hist(accepted_eps, bins=40, density=True, alpha=0.5, color='tab:orange',
            label=f'{len(accepted_eps)} accepted')
    ax.axvline(mu_true, color='red', ls='--', alpha=0.5)
    ax.set_xlabel(r'$\mu$')
    ax.set_title(rf'$\varepsilon = {eps}$')
    ax.legend(fontsize=8)

axes[0].set_ylabel('Density')
fig.suptitle(r'The $\varepsilon$ trade-off: tighter tolerance → better posterior, fewer samples', fontsize=13)
plt.tight_layout()
plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
<h3>§1.3 What this demonstrates</h3>

Three things to take away:

<ol>
<li>When the summary statistic is sufficient (the sample mean, for a Gaussian with known $\sigma$), ABC recovers the true posterior well. When it is not sufficient (median, max), the ABC posterior is wider, shifted, or both. Information has been irreversibly discarded.</li>
<li>The tolerance $\varepsilon$ controls a bias-variance trade-off. Large $\varepsilon$ accepts many samples but the posterior is smoothed (biased wide). Small $\varepsilon$ gives a posterior closer to the truth but accepts almost nothing. There is no free lunch.</li>
<li>This was a one-dimensional problem with 20 data points. For a 1000-pixel spectrum, the acceptance rate at any reasonable $\varepsilon$ is effectively zero unless you compress to a handful of summary statistics, at which point you are back to problem (1).</li>
</ol>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
<h1>§2 Neural SBI</h1>

To put the extension from ABC to SBI simply, rather than comparing simulated and observed data via a hand-crafted summary statistics, you instead train a neural network to learn the relevant mapping from $(\mathbf{\theta}, \mathbf{x})$ simulation pairs. The network thus replaces both the summary statistic and the accept/reject step. 

Before getting into the methods, some useful terminology from <a href="https://doi.org/10.1073/pnas.1912789117">Cranmer, Brehmer &amp; Louppe (2020)</a> (I'd recommend this for further reading). A simulator is a probabilistic program: it takes parameters $\theta$, samples a series of latent variables $z_i \sim p_i(z_i | \theta, z_{<i})$, and produces data $\mathbf{x}$ as output. 

For the stellar population synthesis example from §0, $\theta$ might be the star-formation history and metallicity, while the latent variables $z_i$ are the individual stellar masses drawn from the IMF, the evolutionary tracks sampled for each star, the dust geometry along each line of sight, and so on. 

The likelihood is intractable because evaluating it requires marginalising over every possible execution trace:

$$p(\mathbf{x} | \mathbf{\theta}) = \int p(\mathbf{x}, \mathbf{z} | \theta ) \, d\mathbf{z} $$

For real simulators with large latent spaces, this integral is impossible to compute. Models like this are called implicit models. You can sample from them (run the simulator), but you cannot evalute the likelihood. This is in contrast to prescribed models, where the likelihood can be written down and evaluated. 

There are three main types of neural SBI, distinguished by *what* the network learns to estimate.

<h2>§2.1 Neural Posterior Estimation (NPE)</h2>

NPE trains a neural network $q_\phi(\theta | \mathbf{x})$ to directly approximate the posterior. 

*NOTE: Here, $q$ denotes an approximation to the true distribution $p$. The subscript $\phi$ denotes the train able parameters (weights and biases) of the neural network, so  $q_\phi(\theta | \mathbf{x})$ reads as "the networks approximate posterior over $\theta$ given $\mathbf{x}$, parameterised by $\phi$". I know it can be kind of dense, but I encourage you to actually think what each term means when you come across notation like that.*

The training data is a set of pairs $\{(\theta_i, \mathbf{x_i})\}$, where $\theta_i \sim \pi(\theta)$ (i.e. drawn from the prior) and $\mathbf{x}_i \sim p(\mathbf{x}|\theta_i)$. The network is trained to maximise the log-probability it assigns to the true $\theta_i$ given $\mathbf{x}$:


$$\phi^* = \arg\max_\phi \sum_i \log q_\phi(\theta_i \mid \mathbf{x}_i)$$

Once trained, inference on a new observation $\mathbf{x}_\text{obs}$ is a single forward pass of the Neural Network. This is what makes NPE fully amortised.

The network $q_\phi$ must be a <i>conditional density estimator</i>: given $\mathbf{x}$, it outputs not a point estimate but a full probability distribution over $\theta$. The most common choice is a <i>normalising flow</i>, which we will cover in §3.

<h2>§2.2 Neural Likelihood Estimation (NLE)</h2>

Instead of learning the posterior directly, NLE trains a network to approximate the likelihood: $q_\phi(\mathbf{x} | \theta) \approx p(\mathbf{x} | \theta)$. You then combine this learned likelihood with your prior using standard methods (MCMC, nested sampling, etc) to obtain the posterior. 

This has a practical advantage for problems with multiple independent observations. Suppose you observe 100 stars, each producing a spectrum $\mathbf{x}_i$, and you want to infer a shared parameter $\theta$ (e.g. a global metallicity or an IMF slope). Because the observations are independent, the total likelihood factorises: $\mathcal{L}(\theta) = \prod_i q_\phi(\mathbf{x}_i | \theta)$. The network only needs to learn the likelihood for a <i>single</i> observation. You can then multiply together as many as you like at inference time.

NPE cannot do this. NPE learns the mapping from data to posterior, so it needs to see the full dataset as input during training. If you trained on 100-star datasets and then observe 101 stars, you need to retrain. NLE does not have this problem because the product rule handles any number of observations automatically.

The downside is that NLE is not fully amortised. You still need to run MCMC (or similar) for each new observation to convert the learned likelihood into a posterior. This is usually much cheaper than the original problem (because the network evaluation is fast), but it is not free.

<h2>§2.3 Neural Ratio Estimation (NRE)</h2>

NRE learns the likelihood-to-evidence ratio:

$$ r(\theta, \mathbf{x}) = \frac{p(\mathbf{x} | \theta)}{p(\mathbf{x})} $$

which is proportional to the posterior-to-prior ratio. This is done by training a classifier to distinguish between pairs  $(\theta, \mathbf{x})$ drawn from the joint $p(\theta, \mathbf{x}) = p(\mathbf{x}|\theta)\pi(\theta)$ versus pairs drawn from the product of marginals $\pi(\theta)p(\mathbf{x})$.

Like NLE, you then use MCMC to obtain posterior samples. The advantage is that NRE reduces to a classification problem, which can be easier to train than the generative models required by NPE and NLE.

We won't be covering NRE, but it might be good to know it exists.

<h2>§2.4 Sequential methods</h2>

All three of the above methods can be run in sequential mode (SNPE, SNLE, SNRE). Instead of drawing all training parameters from the priors, you iteratively focus simulations on regions of the parameter space that is relevant for a specific observation $\mathbf{x}_{\text{obs}}$:

1. Run a first round with $\theta \sim \pi(\theta)$, train the network and obtain an initial posterior estimate.
2. Use this estimate as the proposal for the next round of simulations
3. Retrain, refine, repeat

This trades amortisation for simulation efficiency: the resulting network is specialised to one observation rather than being general purpose, but it requires far fewer simulations to achieve a given accuracy. This matters when simulation is expensive, such as in cosmology. 

<h2>§2.5 Aside: Embedding networks</h2>

Above, we saw that ABC requires hand-crafted summary statistics, and that choosing them badly can throw away information. (Most) Neural SBI's solve this by learning the compression within the neural network.

If you are familiar with autoencoders, this is the same idea as the embedding part of the autoencoder. If you are not familiar with autoencoders, the embedding network is simply a neural network that starts with a certain dimensionality, and as more layers are added, the dimensionality is decreased. E.g:

$$ \text{input} \rightarrow 128\text{ dim} \rightarrow 64\text{ dim} \rightarrow 32\text{ dim} \rightarrow \text{output}$$ 

When the data $\mathbf{x}$ is high-dimensional (a spectrum, image, etc) the density estimator does not act on $\mathbf{x}$ directly. Instead, an embedding network $h_\varphi(\mathbf{x})$ compresses it to a low-dimensional representation, and the density estimator operates on the output of the embedding network.

Importantly, the embedding network is trained together with the density estimator, end-to-end, so it learns to extract the features that are informative about $\theta$.

<h2>§2.6 Which one do I use?</h2>

For a first pass: use NPE. It is the simplest to set up, fully amortised, and the default recommendation in the <code>sbi</code> Python package (<a href="https://sbi-dev.github.io/sbi/">Boelts et al. 2025</a>). NLE becomes attractive when you have many i.i.d. observations from the same model, or when you want to combine the learned likelihood with informative priors that differ from the training prior. NRE is useful when you only need the likelihood ratio (e.g. for hypothesis testing) or when training a generative model is difficult. Sequential variants are worth considering when simulations are expensive and you only care about one observation.

All three methods share the same prerequisite: a conditional density estimator powerful enough to represent the distribution being learned. The workhorse for this is the normalising flow.

<h2>§2.7 Caveats</h2>

Two things to keep in mind:

<ol>
<li>The simulator is the model. The neural network faithfully learns whatever distribution the simulator produces. If the simulator is wrong (missing physics, incorrect noise model, numerical artefacts), the network will learn the wrong posterior, confidently. </li>
<li>Diagnostics are essential. Because SBI gives you no guarantee that the learned posterior is correct, you must validate it. The standard tools are:
<ul>
<li><i>Posterior predictive checks</i>: draw $\theta$ from the learned posterior, simulate data, check that it looks like the observation.</li>
<li><i>Simulation-based calibration</i> (SBC): over many test simulations, check that the posterior credible intervals have the correct coverage (e.g. the 90% interval contains the true $\theta$ 90% of the time).</li>
<li><i>Coverage diagnostics</i>: check calibration both globally (averaged over the prior) and locally (for specific observations of interest).</li>
</ul>
If these checks fail, the network may need more training data, a more expressive architecture, or the simulator may be misspecified. We will cover diagnostics in detail later.</li>
</ol>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
<h1>§3 Neural Density Estimators</h1>
<h2>§3.1 How do you build a NN whose output is a density?</h2>

The piece of machinery sitting at the heart of modern SBI is a <b>neural density estimator</b> (NDE): a neural network whose <i>output</i> is a probability density. Specifically, a network with parameters $\phi$ that maps an input $\mathbf{x}$ to a conditional density $q_\phi(\theta \mid \mathbf{x})$ over $\theta$. Once you have one of these, you train it by maximising the likelihood of simulator draws,$$\phi^* = \arg\max_\phi \sum_i \log q_\phi(\theta_i \mid \mathbf{x}_i),$$and the trained $q_{\phi^*}(\theta \mid \mathbf{x}_\text{obs})$ is your posterior approximation.

There are several types of Neural Density Estimators (NDEs). Here we will look at two common ones:

<ul>
<li>Mixture density networks (MDNs): the network outputs the parameters of a Gaussian mixture model (i.e. a set of means and variances). Simple, interpretable, but limited expressivity.</li>
<li>Normalising flows: the network defines an invertible transformation between a simple base distribution and the target density. Much more expressive and is the workhorse of modern SBI.</li>
</ul>


</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
<h2>§3.2 What you need to know about neural networks</h2>

A neural network is a parameterised function $f_\phi(\mathbf{x})$ with trainable weights $\phi$. Training means adjusting $\phi$ to minimise a loss function. In our case, the loss is the negative log-probability the network assigns to the true parameters:

$$\mathcal{L}(\phi) = -\frac{1}{N} \sum_{i=1}^{N} \log q_\phi(\theta_i \mid \mathbf{x}_i)$$

The <code>sbi</code> package handles training internally (optimiser selection, batching, convergence checks, etc.). You will not need to write a training loop, define an optimiser, or subclass <code>nn.Module</code>. What you <i>do</i> need to understand is what the network is outputting: not a point estimate, but a <b>probability distribution</b>. A standard neural network takes an input and returns a number (or a vector of numbers). A neural density estimator takes an input and returns the parameters of a probability distribution. The next two sections build this intuition with a concrete example.</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">
    
<h2>§3.3 Mixture Density Networks</h2>

We want a function $q_\phi(\theta \mid \mathbf{x})$ that is a valid probability density in $\theta$, parametrically dependent on $\mathbf{x}$, and flexible enough to capture whatever shape the true posterior has. The trick of an MDN is to fix the family of $q_\phi$ to a Gaussian mixture, and let the neural network output the parameters of that mixture as a function of $\mathbf{x}$:

$$q_\phi(\theta \mid \mathbf{x}) = \sum_{k=1}^{K} w_k(\mathbf{x}) \; \mathcal{N}\!\left(\theta \;\big|\; \boldsymbol{\mu}_k(\mathbf{x}), \boldsymbol{\Sigma}_k(\mathbf{x})\right)$$

The network takes $\mathbf{x}$ as input and produces three things: $K$ mixture weights $w_k(\mathbf{x})$, $K$ means $\boldsymbol{\mu}_k(\mathbf{x})$, and $K$ covariance matrices $\boldsymbol{\Sigma}_k(\mathbf{x})$. The weights are passed through softmax (so they are positive and sum to 1); the covariance entries are parameterised so that $\boldsymbol{\Sigma}_k$ is guaranteed positive-definite. Given $N$ training pairs $(\theta_i, \mathbf{x}_i)$, the loss is the negative log-likelihood:

$$\mathcal{L}(\phi) = -\frac{1}{N} \sum_{i=1}^{N} \log q_\phi(\theta_i \mid \mathbf{x}_i)$$

This is differentiable in $\phi$, so standard gradient-based optimisation applies.

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">Why mixtures specifically?</summary><div style="margin-top: 10px;">A single Gaussian only captures unimodal, elliptical posteriors. Many real posteriors are multimodal (period aliases, label-switching, etc.) or have curved degeneracies that no single Gaussian can represent. Gaussian mixtures with enough components can in principle approximate any sufficiently smooth density. The price is that you have to choose $K$, and that very flexible mixtures can be hard to train (mode collapse, near-degenerate components). This is part of why normalising flows have largely replaced MDNs in current SBI codebases.</div></details></div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§3.4 A 1D MDN, end to end</h2>

Let's build the smallest possible MDN to make this concrete. We'll fit a 1-parameter "posterior" to a synthetic dataset where the conditional $p(\theta \mid x)$ is bimodal, so a single Gaussian can't possibly fit it. 

Don't worry too much about the details of the code, the `sbi` package does a lot of this behind the scenes.

Generate $(\theta, x)$ pairs:
</div>

In [ ]:
N = 5000
torch.manual_seed(0)
theta_train = torch.empty(N, 1).uniform_(-1, 1)
x_train = theta_train**2 + 0.05 * torch.randn(N, 1)

fig, ax = plt.subplots(1, 1, figsize=(6, 4))
ax.scatter(x_train.numpy(), theta_train.numpy(), s=2, alpha=0.3)
ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$\theta$")
ax.set_title(r"Joint samples — for each $x>0$ there are two values of $\theta$")
plt.tight_layout(); plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

The forward map $\theta \mapsto x = \theta^2$ is two-to-one, so the inverse $p(\theta \mid x)$ has two branches at $\pm\sqrt{x}$ for any $x > 0$. Now define an MDN with $K=5$ Gaussian components:

</div>

In [ ]:
# A minimal 1D MDN: outputs (logit_w, mu, log_sigma) for K Gaussian components
class MDN1D(nn.Module):
    def __init__(self, K=5, hidden=64):
        super().__init__()
        self.K = K
        self.body = nn.Sequential(
            nn.Linear(1, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        # Three heads — one per Gaussian parameter type
        self.head_logit_w   = nn.Linear(hidden, K) # weight
        self.head_mu        = nn.Linear(hidden, K) # mean
        self.head_log_sigma = nn.Linear(hidden, K) # sigma

    def forward(self, x):
        h = self.body(x)
        log_w   = torch.log_softmax(self.head_logit_w(h), dim=-1)  # log mixture weights
        mu      = self.head_mu(h)
        log_sig = self.head_log_sigma(h).clamp(min=-7.0, max=2.0)  # numerical safety
        return log_w, mu, log_sig

In [ ]:
# Negative log-likelihood loss for the MDN
def mdn_nll(log_w, mu, log_sig, theta):
    # theta: (N, 1); broadcast over K components
    sig = log_sig.exp()
    # log N(theta | mu_k, sig_k^2)
    log_probs = -0.5 * ((theta - mu) / sig)**2 - log_sig - 0.5 * np.log(2*np.pi)
    # log sum_k w_k * N(...)  =  logsumexp_k (log w_k + log N)
    log_q = torch.logsumexp(log_w + log_probs, dim=-1)
    return -log_q.mean()

In [ ]:
# Train
mdn = MDN1D(K=5, hidden=64)
opt = torch.optim.Adam(mdn.parameters(), lr=2e-3)
losses = []
for step in tqdm(range(3000)):
    opt.zero_grad()
    log_w, mu, log_sig = mdn(x_train)
    loss = mdn_nll(log_w, mu, log_sig, theta_train)
    loss.backward(); opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(losses); ax.set_xlabel("step"); ax.set_ylabel("NLL")
ax.set_title("MDN training loss")
plt.tight_layout(); plt.show()
print(f"Final NLL: {losses[-1]:.3f}")

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Now evaluate the trained $q_\phi(\theta \mid x)$ at three values of $x$, showing the bimodal posterior at each, with mode separation $2\sqrt{x}$:

</div>

In [ ]:
# Evaluate q(theta | x) at three values of x — note bimodality at x>0
@torch.no_grad()
def mdn_density(x_val, theta_grid):
    x_b = torch.full((len(theta_grid), 1), float(x_val))
    log_w, mu, log_sig = mdn(x_b)
    sig = log_sig.exp()
    th = theta_grid.unsqueeze(-1)  # (N_grid, 1)
    log_probs = -0.5 * ((th - mu) / sig)**2 - log_sig - 0.5 * np.log(2*np.pi)
    log_q = torch.logsumexp(log_w + log_probs, dim=-1)
    return log_q.exp()

theta_grid = torch.linspace(-1.2, 1.2, 400)
fig, axes = plt.subplots(1, 3, figsize=(12, 3.2), sharey=True)
for ax, x_val in zip(axes, [0.1, 0.4, 0.8]):
    q = mdn_density(x_val, theta_grid).numpy()
    ax.plot(theta_grid.numpy(), q, lw=2)
    ax.set_xlabel(r"$\theta$"); ax.set_title(rf"$q_\phi(\theta \mid x={x_val})$")
axes[0].set_ylabel("density")
plt.tight_layout(); plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

Look at the scatter plot above, and imagine the histogram that would be drawn for $x = [0.1, 0.4, 0.8]$ - this should align with these posteriors above. 

Notice how the learned density correctly captures both branches $\theta = \pm\sqrt{x}$ for $x > 0$, with the modes spreading and merging as $x \to 0$. A single Gaussian could not represent this: it has one mean and one width, no chance of two separate peaks.

This is, in miniature, what NPE does. Replace this 1D toy with a real simulator that maps $\theta \in \mathbb{R}^d$ to $\mathbf{x} \in \mathbb{R}^D$, replace the MDN with a normalising flow conditional on $\mathbf{x}$, train on $10^4$–$10^6$ simulator draws, and evaluate on your real observation $\mathbf{x}_\text{obs}$. That's the whole algorithm.


<h2>§3.5 Why MDNs are not enough for real problems</h2>

MDNs work for low-dimensional $\theta$, but they have three fundamental limitations:

<ol>
<li>You have to pick $K$. Too small, you can't capture the posterior. Too large, training becomes unstable (degenerate components, mode collapse).</li>
<li>Covariance parameterisation gets ugly in high $d$. A full $d \times d$ Cholesky factor per mixture component scales as $d^2$. Diagonal covariances are cheap but lose all correlation structure.</li>
<li>Tail behaviour is locked into Gaussian. Mixtures of Gaussians have Gaussian tails. Real posteriors can have very different tail behaviour (power-law, bounded, etc.).</li>
</ol>

The fix for all three is to switch to a more expressive density family. That is what normalising flows give us.

<b>Bottom line:</b> MDNs are useful as a teaching example and still adequate for some low-dimensional problems, but every modern SBI codebase reaches for normalising flows by default.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§4. Normalising Flows</h1>

<h2>§4.0 The idea</h2>

The MDN approach works by having the network output the parameters of a fixed family of distributions (a mixture of Gaussians). The limitation is that you're stuck with that family. Normalising flows are more expressive because they take a different approach. They start with a simple distribution you know how to work with (usually a standard Gaussian), and <i>warp</i> it into whatever complicated shape you need.

A normalising flow is a parameterised, invertible function $f_\phi$ that does this warping. You draw a sample $z$ from a standard Gaussian, push it through $f_\phi$, and out comes a sample $\theta = f_\phi(z)$ from the target distribution. The function $f_\phi$ is a neural network (with train able weights $\phi$), but a special one: it is designed to be invertible, so you can also go backwards from $\theta$ to $z$.

If you warp samples from a known distribution through some function, what is the density of the warped samples? This is answered by the change-of-variables formula, which you may have seen in a different context. In 1D, if $z \sim p_z(z)$ and $\theta = f(z)$, the density of $\theta$ is:

$$q(\theta) = p_z(z) \cdot \left|\frac{dz}{d\theta}\right|$$
The first factor asks: what was the base density at the point $z$ that maps to this $\theta$? The second factor is the Jacobian, which corrects for the stretching or compression that $f$ applies. If $f$ stretches a region of $z$-space (spreading samples apart), the density must decrease to compensate, and vice versa.

In multiple dimensions, the absolute derivative becomes the absolute determinant of the Jacobian matrix:


$$q_\phi(\boldsymbol{\theta}) = p_z(\mathbf{z}) \cdot \left|\det \frac{\partial \mathbf{z}}{\partial \boldsymbol{\theta}}\right|$$

This gives you two things for free:

<ol>
<li>Sampling: draw $z \sim p_z$ (easy, it's a Gaussian), compute $\theta = f_\phi(z)$. </li>
<li>Density evaluation: given a $\theta$, compute $z = f_\phi^{-1}(\theta)$, evaluate $p_z(z)$, multiply by the Jacobian determinant. This tells you $q_\phi(\theta)$.</li>
</ol>

Both operations require the function to be invertible and the Jacobian determinant to be cheap to compute. How to build such functions is the subject of §4.1.

<h2>§4.0.1 Making it conditional</h2>

For SBI we don't want an unconditional density $q_\phi(\theta)$. We want a <i>conditional</i> density $q_\phi(\theta \mid \mathbf{x})$: given this particular dataset $\mathbf{x}$, what is the posterior over $\theta$? The trick is simple: make the warping function depend on $\mathbf{x}$. The flow becomes $f_\phi(z; \mathbf{x})$, and the density becomes:

$$q_\phi(\boldsymbol{\theta} \mid \mathbf{x}) = p_z(\mathbf{z}) \cdot \left|\det \frac{\partial \mathbf{z}}{\partial \boldsymbol{\theta}}\right|$$

Here, $z=f^{-1}_\phi(\theta;x) $.

Training is the same as the MDN: maximise $\sum_i \log q_\phi(\theta_i \mid \mathbf{x}_i)$ over the simulator draws. The loss function is identical; only the density estimator has changed.

<h2>§4.0.2 Why bother?</h2>

The reason flows have largely replaced MDNs is that they have <i>no parametric restriction on shape</i>. An MDN can only produce mixtures of Gaussians: elliptical blobs, Gaussian tails, and you have to choose $K$. A flow composed of many invertible layers can in principle warp a Gaussian into any smooth density: heavy tails, sharp ridges, multimodality, curved degeneracies. You don't choose a $K$, you just stack more layers for more expressivity.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.1 Two operations, both cheap</h2>

A neural network alone is usually not invertible, it is many-to-one in general, and computing $f^{-1}$ is intractable. For flow design you need to construct parameterised functions that are:

<ol>
<li>Invertible (so you can map between $\theta$-space and base-space in both directions),</li>
<li>Have a tractable Jacobian determinant (usually triangular), so $\det J$ is just the product of diagonal entries.</li>
</ol>

Different flow architectures achieve this in different ways. The two common ones are:

<ul>
<li>Coupling flows (RealNVP, NICE): split $\theta$ into two halves; transform one half conditionally on the other using an arbitrary neural net. The Jacobian is triangular by construction. Invertible because you can recover the transformed half by undoing the same operation.</li>
<li>Autoregressive flows (MAF, IAF): transform each dimension conditionally on the previous ones. Same triangular Jacobian. MAF (Masked Autoregressive Flow, <a href='https://arxiv.org/abs/1705.07057'>Papamakarios et al. 2017</a>) is the default density estimator in <code>sbi</code>.</li>
</ul>

A more recent architecture, neural spline flows (NSF, <a href='https://arxiv.org/abs/1906.04032'>Durkan et al. 2019</a>), uses monotonic rational-quadratic splines as the elementwise transformations, which gives much more expressive layers than affine transforms. NSF has become a popular alternative to MAF when more expressivity is needed.

The mechanics matter less than the consequences: with these architectures you get a function whose density and inverse are both cheap to evaluate, and you can stack many of them to compose more expressive flows.

<details class="alert alert-info" style="margin: 10px 0;"><summary style="cursor: pointer; font-weight: bold;">If you want more details </summary><div style="margin-top: 10px;">
The canonical review is <a href='https://arxiv.org/abs/1912.02762'>Papamakarios et al. (2021), "Normalising Flows for Probabilistic Modelling and Inference"</a>. It covers all of the architectures, the universal-approximation properties, and practical training tricks. About 60 pages, dense but probably handy (it is on my to read pile).
</div></details>

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.2 Watching a flow warp a Gaussian</h2>

To make this concrete, let's train a small unconditional MAF on a 2D target (the classic "two moons") and visualise how the Gaussian base distribution is progressively warped to match it.

We use <code>zuko</code>, a lightweight modern flow library that integrates well with PyTorch. (The <code>sbi</code> package can use <code>zuko</code> as a backend; you don't <i>need</i> <code>zuko</code> to do SBI, but it's the cleanest small example.)

</div>

In [ ]:
# zuko was imported above; verify version - mine is 1.5.0
print(f"zuko version: {zuko.__version__}")

In [ ]:
# Generate "two moons" target data
def two_moons(n, noise=0.05):
    """Standard sklearn-style two-moons dataset."""
    n1 = n // 2; n2 = n - n1
    t1 = torch.rand(n1) * np.pi
    x1 = torch.stack([torch.cos(t1), torch.sin(t1)], dim=-1)
    t2 = torch.rand(n2) * np.pi
    x2 = torch.stack([1 - torch.cos(t2), 0.5 - torch.sin(t2)], dim=-1)
    data = torch.cat([x1, x2], dim=0)
    data = data + noise * torch.randn_like(data)
    # Centre and scale a bit for nicer plotting
    data = (data - data.mean(0)) / data.std(0)
    return data

torch.manual_seed(0)
moons = two_moons(5000)
fig, ax = plt.subplots(1, 1, figsize=(5, 4))
ax.scatter(moons[:, 0], moons[:, 1], s=2, alpha=0.4)
ax.set_aspect('equal')
ax.set_title("two moons target")
plt.tight_layout();
plt.show()

In [ ]:
# Build a small MAF with 5 transform layers
flow = zuko.flows.MAF(features=2, transforms=5, hidden_features=(64, 64))

# Train by maximum likelihood
opt = torch.optim.Adam(flow.parameters(), lr=2e-3)
losses = []
for step in tqdm(range(2000)):
    opt.zero_grad()
    loss = -flow().log_prob(moons).mean()
    loss.backward(); opt.step()
    losses.append(loss.item())

fig, ax = plt.subplots(1, 1, figsize=(6, 3))
ax.plot(losses); ax.set_xlabel("step"); ax.set_ylabel("NLL")
ax.set_title("flow training loss")
plt.tight_layout(); plt.show()
print(f"Final NLL: {losses[-1]:.3f}")

In [ ]:
# Build and train, saving density snapshots at intervals
flow = zuko.flows.MAF(features=2, transforms=5, hidden_features=(64, 64))
opt = torch.optim.Adam(flow.parameters(), lr=2e-3)

snapshot_steps = [0, 25, 100, 300, 800, 2000]
snapshots = {}

@torch.no_grad()
def density_grid(flow, lim=2.5, n=200):
    g = torch.linspace(-lim, lim, n)
    X, Y = torch.meshgrid(g, g, indexing='xy')
    pts = torch.stack([X.flatten(), Y.flatten()], dim=-1)
    log_q = flow().log_prob(pts).reshape(n, n)
    return X, Y, log_q.exp()

for step in tqdm(range(2001)):
    if step in snapshot_steps:
        snapshots[step] = density_grid(flow)
    
    opt.zero_grad()
    loss = -flow().log_prob(moons).mean()
    loss.backward(); opt.step()

fig, axes = plt.subplots(1, len(snapshot_steps), figsize=(3.2 * len(snapshot_steps), 3.2))
for ax, step in zip(axes, snapshot_steps):
    X, Y, dens = snapshots[step]
    ax.pcolormesh(X.numpy(), Y.numpy(), dens.numpy(), shading='auto', cmap='viridis')
    ax.scatter(moons[:, 0], moons[:, 1], s=0.3, alpha=0.06, color='white')
    ax.set_aspect('equal')
    ax.set_xlim(-2.5, 2.5); ax.set_ylim(-2.5, 2.5)
    ax.set_title(f'Step {step}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

fig.suptitle(r'Flow learning to approximate the two-moons density', fontsize=12, y=1.02)
plt.tight_layout(); plt.show()

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§4.3 What's happening under the hood</h2>

The flow took the standard Gaussian base distribution and progressively bent it into the moons shape via 5 invertible affine-coupling layers (with neural-net-parameterised scale and shift). Each layer is a small, invertible warp. The composition is a complicated invertible function. 

This is the architecture that powers most modern NPE. For a conditional flow used in SBI, the only change is that the neural networks producing the per-layer scale and shift parameters take $\mathbf{x}$ as an additional input. So the flow becomes $\theta = f_\phi(z; \mathbf{x})$, and you train on simulator draws as before.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§5. SBI in Practice: the <code>sbi</code> package</h1>

Everything above was built from scratch to show you what is happening under the hood. In practice, you use the <code>sbi</code> Python package (<a href="https://sbi-dev.github.io/sbi/">Boelts et al. 2025</a>), which wraps all of this into a clean API.

The workflow is:
<ol>
<li>Define a simulator: a function that takes a parameter vector $\theta$ and returns synthetic data $\mathbf{x}$.</li>
<li>Define a prior over $\theta$.</li>
<li>Choose an inference method (NPE, NLE, or NRE) and train.</li>
<li>Pass your real observation to get a posterior.</li>
</ol>

Let's do this end-to-end on a toy problem: inferring the mean and standard deviation of a Gaussian from a small sample. This is a problem where we <i>know</i> the answer, so we can check that SBI gets it right.

</div>

In [ ]:
##############################
# §5.1 Define the simulator  #
##############################

# Ground truth (hidden from the network)
theta_true = torch.tensor([2.0, 0.5])  # [mu, sigma]
n_obs = 20

# The simulator: given (mu, sigma), draw n_obs samples from N(mu, sigma)
# and return the raw data vector as x
def simulator(theta):
    mu, sigma = theta
    return mu + sigma * torch.randn(n_obs)

# Generate the "observed" data once
torch.manual_seed(0)
x_obs = simulator(theta_true)

fig, ax = plt.subplots(figsize=(6, 3))
ax.hist(x_obs.numpy(), bins=10, density=True, alpha=0.5, color="tab:blue")
ax.axvline(theta_true[0].item(), color="red", ls="--", label=rf"true $\mu = {theta_true[0].item()}$")
ax.set_xlabel("x")
ax.set_ylabel("density")
ax.set_title(f"Observed data ({n_obs} points)")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
##############################################
# §5.2 Define prior, build and train the NPE #
##############################################

# Prior: uniform over mu in [-5, 5] and sigma in [0.1, 3.0]
prior = BoxUniform(
    low=torch.tensor([-5.0, 0.1]),
    high=torch.tensor([5.0, 3.0])
)

# Build the NPE inference object
inference = NPE(prior=prior)

# Simulate training data: draw theta from prior, run simulator
n_train = 10_000
theta_train = prior.sample((n_train,))
x_train = torch.stack([simulator(t) for t in tqdm(theta_train, desc="Simulating")])

# Append simulations and train
inference.append_simulations(theta_train, x_train)
density_estimator = inference.train(show_train_summary=True)


In [ ]:
#############################
# §5.3 Get the posterior    #
#############################

# Build the posterior object and sample from it
posterior = inference.build_posterior(density_estimator)

# This is the amortised bit: a single forward pass gives you the posterior
samples = posterior.sample((10_000,), x=x_obs)

# Plot
fig, axes = pairplot(
    samples.numpy(),
    limits=[[-5, 5], [0.1, 3.0]],
    labels=[r"$\mu$", r"$\sigma$"],
    truths=theta_true.numpy(),
    figsize=(5, 5),
)
fig.suptitle("NPE posterior (sbi package)", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print(f"True theta:     mu = {theta_true[0].item():.2f}, sigma = {theta_true[1].item():.2f}")
print(f"Posterior mean: mu = {samples[:, 0].mean():.2f}, sigma = {samples[:, 1].mean():.2f}")
print(f"Posterior std:  mu = {samples[:, 0].std():.2f},  sigma = {samples[:, 1].std():.2f}")


<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h2>§5.4 What just happened</h2>

In about 20 lines of code, we:
<ol>
<li>Defined a simulator (a function that takes $\theta$ and returns data).</li>
<li>Defined a prior (a <code>BoxUniform</code> distribution).</li>
<li>Generated 10,000 training simulations.</li>
<li>Trained a conditional normalising flow to learn $q_\phi(\theta \mid \mathbf{x})$.</li>
<li>Obtained 10,000 posterior samples for our observation in a single forward pass.</li>
</ol>

Notice that we never wrote down a likelihood. The simulator <i>is</i> the likelihood. The <code>sbi</code> package handled the flow architecture, training loop, and density evaluation. The posterior should recover the true values $\mu = 2.0$, $\sigma = 0.5$ within the expected uncertainty.

This is the workflow you will use in the tutorial session, applied to more interesting simulators.

</div>

<div style="max-width: 800px; margin: 0 auto; text-align: justify;">

<h1>§6. What this notebook did not cover</h1>

This notebook introduced the core ideas. Several important topics were deliberately omitted to keep the reading manageable, and will appear in the tutorial session or as further reading:

<ul>
<li><b>Simulation-based calibration (SBC)</b>: the standard diagnostic for checking whether your learned posterior has correct coverage. Essential in practice. We will cover this in the tutorial.</li>
<li><b>Sequential methods (SNPE, SNLE)</b>: mentioned in §2.4 but not demonstrated. These iteratively refine the proposal distribution to focus simulations around a specific observation.</li>
<li><b>Embedding networks</b>: mentioned in §2.5. When data is high-dimensional (images, spectra), you need a neural network to compress it before the density estimator sees it. The <code>sbi</code> package supports custom embedding networks.</li>
<li><b>Model misspecification</b>: what happens when the simulator does not match reality. This is the central open problem in SBI. Diagnostics help, but there is no silver bullet.</li>
<li><b>Neural Ratio Estimation (NRE)</b>: the classification-based approach from §2.3. Useful for hypothesis testing and when generative modelling is hard.</li>
<li><b>Comparison with classical methods</b>: when should you use SBI instead of MCMC or nested sampling? Rule of thumb: if you can write a likelihood and evaluate it cheaply, use MCMC. If you cannot, or if you need amortisation over many observations, use SBI.</li>
</ul>

<h2>§6.1 Further reading</h2>

<ul>
<li><b>Cranmer, Brehmer &amp; Louppe (2020)</b>, <i>The frontier of simulation-based inference</i>, PNAS 117(48). The standard review paper. Covers NPE, NLE, NRE, and the taxonomy of simulator types.</li>
<li><b>Papamakarios, Nalisnick, Rezende, Mohamed &amp; Lakshminarayanan (2021)</b>, <i>Normalizing Flows for Probabilistic Modeling and Inference</i>, JMLR 22(57). The canonical review of normalising flows (~60 pages).</li>
<li><b>Lueckmann, Boelts, Greenberg, Gonçalves &amp; Macke (2021)</b>, <i>Benchmarking Simulation-Based Inference</i>, AISTATS. Systematic comparison of NPE, NLE, NRE on benchmark problems.</li>
<li><a href="https://sbi-dev.github.io/sbi/"><code>sbi</code> documentation</a> (Boelts et al. 2025). Tutorials, API reference, and worked examples.</li>
<li><a href="https://github.com/sbi-benchmark/sbibm"><code>sbibm</code></a>: a benchmark suite for SBI methods, useful for testing new approaches.</li>
</ul>

</div>